# 上下文和记忆

## 上下文工程

在 LangChain 中， 记忆(Memory) 就是专门负责“ 存储历史交互信息 ”的组件，核心作用是「 保存上下文 」和「 提供上下文 」，让LLM在每次响应时，都能“看到”之前的对话内容。

上下文工程(Context Engineering) 负责 “合理组织”这些记忆和任务信息 ，让LLM的响应更连贯、更贴合需求。这也是Agent能实现 复杂多轮交互 的核心基础。

![image.png](images/context.png)

## 记忆的分类

- 短期记忆 ：会话级记忆，作用范围是单个对话线程内，一旦开始新对话(更换线程id)，记忆即消失
- 长期记忆：跨会话级记忆，在会话间存储用户特定或应用级数据， 并在 会话线程间共享 。它可以随时在任何线程中被调用。记忆的范围是任意自定义命名空间，而不仅仅是单一线程 ID。

## 记忆的管理

在LangChain v1.x版本中，Agent是构建在LangGraph图结构之上的，通过上文提到的state和store构建记忆系统。使用更简单、功能更统一。
- state：短期记忆对象，以 会话 为单位组织，包含当前会话的所有消息记录以及自定义信息。
- store：长期记忆对象， 跨会话持久化 的数据，通常需要结合向量数据库或外部存储实现。

# 短期记忆

LangChain1.x 的短期记忆是三者的组合：
State（会话内部状态） + Checkpointer（持久化机制） + Thread ID（会话作用域）

- State ：默认 存储历史消息列表messages ，通过State 管理历史消息
- Checkpointer ：负责将State 作为检查点持久化保存，检查点是某个时刻的State 快照
- Thread ID ：用于唯一标识State ，LangChain运行时会按照 thread_id 读写State快照

### 基于内存的持久化器

In [1]:
from anyio.lowlevel import checkpoint
# 没有记忆
from langchain.agents import  create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)
model=init_chat_model(model="deepseek-flash",
                      model_provider="deepseek")

agent=create_agent(model=model,
                   tools=[])

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]
})
print(f"Agent: {response1['messages'][-1].content}")
print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]
})
print(f"Agent: {response2['messages'][-1].content}")


第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 我不知道你叫什么名字，因为你还没有告诉我。你可以告诉我你想让我怎么称呼你吗？


In [8]:
# 拥有记忆

from langgraph.checkpoint.memory import InMemorySaver

checkpointer=InMemorySaver()

agent=create_agent(model=model,
                   tools=[],
                   checkpointer=checkpointer #添加内存管理
                     )

config={
    "configurable":{
        "thread_id":"1"
    }
}

print("\n第一轮对话：")
response1 = agent.invoke({
    "messages": [HumanMessage("我叫张三")]},
    config=config
)
print(f"Agent: {response1['messages'][-1].content}")
print("\n第二轮对话：")
response2 = agent.invoke({
    "messages": [HumanMessage("我叫什么？")]},
    config=config
)
print(f"Agent: {response2['messages'][-1].content}")



第一轮对话：
Agent: 你好，张三！很高兴认识你。有什么我可以帮你的吗？

第二轮对话：
Agent: 你叫张三。


### 只需传入 checkpointer 和 config，Agent 就能自然具备连续对话能力。

In [3]:
from rich import print as rprint

latest_state = agent.get_state(config)
rprint(latest_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='我叫张三',
                additional_kwargs={},
                response_metadata={},
                id='4108e897-cfa1-45c4-b22d-9680748397ea'
            ),
            AIMessage(
                content='你好，张三！很高兴认识你。有什么我可以帮你的吗？',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 
'我们需要回答中文。用户只是说“我叫张三”。应该介绍/回应。可以礼貌说你好张三，有什么可以帮你。无需复杂。注意不要泄露
。最终简洁。'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 51,
                        'prompt_tokens': 32,
                        'total_tokens': 83,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 36,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 32
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': 'cc60da07-f95c-412c-b8dc-279eeb19f06c',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a1e979-daa7-705c-9ed5-6b7cd7b26d91-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 32,
                    'output_tokens': 51,
                    'total_tokens': 83,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 36}
                }
            ),
            HumanMessage(
                content='我叫什么？',
                additional_kwargs={},
                response_metadata={},
                id='bf86085c-02f5-4cf5-b788-62f2c7285d02'
            ),
            AIMessage(
                content='你叫张三。',
                additional_kwargs={
                    'refusal': None,
                    'reasoning_content': 
'我们需要回答用户中文问“我叫什么？”根据对话历史，用户之前说“我叫张三”。所以回答：你叫张三。简洁。需要确认。'
                },
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 37,
                        'prompt_tokens': 53,
                        'total_tokens': 90,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 32,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 53
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
                    'id': '0d3236f6-47f0-4776-8ddc-c714224a6234',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a1e979-df8d-729f-8c96-da750071b184-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 53,
                    'output_tokens': 37,
                    'total_tokens': 90,
                    'input_token_details': {'cache_read': 0},
                 

In [9]:
print("\n第三轮对话：")
response3 = agent.invoke({
    "messages": [HumanMessage("我刚才问了什么问题？")]},
    config=config
)
print(response3["messages"][-1].content)


第三轮对话：
你刚才问的是：“我叫什么？”


In [13]:
# 更新线程id，则会重新开始对话

config={
    "configurable":{
        "thread_id":"3"
    }
}

response4=agent.invoke({
    "messages":[HumanMessage("我记得我叫什么吗")]
    },
    config=config
)

print(response4["messages"][-1].content)


我不记得你叫什么。我这边没有你的个人信息，也没有更早的聊天记录；如果你愿意告诉我，我可以在这段对话里这样称呼你。


# 核心步骤

## 1.初始化记忆引擎
    checkpointer=InMemorySaver()创建一个内存级的记忆存储。
    InMemorySaver内存中保存，进程结束就丢失数据，适合测试。生产环境可换成数据库持久化的 SqliteSaver 、 PostgresSaver 等
## 2.绑定 Agent
    在 create_agent 时传入 checkpointer ，让 Agent 具备状态存储能力。
## 3. 设定会话 ID
    通过 config = {"configurable": {"thread_id": "1"}} 为每次调用指定线程标识。同一个 thread_id 共享记忆，不同 thread_id 完全隔离。

thread_id 是记忆管理的核心开关：在会话2里询问会话1的会话信息，Agent 会表示不知道——因为双方记忆空间完全隔离。

### 工作原理

自动追加历史
```
# 你只需要传新消息
agent.invoke(
    {"messages": [{"role": "user", "content": "新问题"}]},
    config
)
```

此时，checkpointer自动： 读取之前的历史，追加新消息，调用模型(传入完整历史)，保存新历史

- 当程序重启时，内存中的历史会丢失

解决方法 ：持久化  SQLite,postgreSQL

- 内存也会为无限增长

默认情况下，InMemorySaver 会保存所有消息。
    问题：
   1. 消息越来越多（无限增长，需要管理上下文
   2. token消耗增加，甚至会超过模型的 token 限制
   3. 响应速度变慢、成本增加

解决方法：上下文管理

- 清空某个会话的历史

临时方案：
   1. 使用新的thread_id
   2. 重新创建agent
